In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

In [16]:
import datasets 
dataset = datasets.load_dataset('ucberkeley-dlab/measuring-hate-speech')   
df = dataset['train'].to_pandas()
df.describe()

,comment_id,annotator_id,platform,sentiment,respect,insult,humiliate,status,dehumanize,violence,...,hatespeech,hate_speech_score,infitms,outfitms,annotator_severity,std_err,annotator_infitms,annotator_outfitms,hypothesis,annotator_age
count,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.00000,135556.000000,135556.000000,135556.000000,135556.000000,...,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135556.000000,135451.000000
mean,23530.416138,5567.097812,1.281352,2.954307,2.828875,2.56331,2.278638,2.698575,1.846211,1.052045,...,0.744733,-0.567428,1.034322,1.001052,-0.018817,0.300588,1.007158,1.011841,0.014589,37.910772
std,12387.194125,3230.508937,1.023542,1.231552,1.309548,1.38983,1.370876,0.898500,1.402372,1.345706,...,0.932260,2.380003,0.496867,0.791943,0.487261,0.236380,0.269876,0.675863,0.613006,11.641276
min,1.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,...,0.000000,-8.340000,0.100000,0.070000,-1.820000,0.020000,0.390000,0.280000,-1.578693,18.000000
25%,18148.000000,2719.000000,0.000000,2.000000,2.000000,2.00000,1.000000,2.000000,1.000000,0.000000,...,0.000000,-2.330000,0.710000,0.560000,-0.380000,0.030000,0.810000,0.670000,-0.341008,29.000000
50%,20052.000000,5602.500000,1.000000,3.000000,3.000000,3.00000,3.000000,3.000000,2.000000,0.000000,...,0.000000,-0.340000,0.960000,0.830000,-0.020000,0.340000,0.970000,0.850000,0.110405,35.000000
75%,32038.250000,8363.000000,2.000000,4.000000,4.000000,4.00000,3.000000,3.000000,3.000000,2.000000,...,2.000000,1.410000,1.300000,1.220000,0.350000,0.420000,1.170000,1.130000,0.449555,45.000000
max,50070.000000,11142.000000,3.000000,4.000000,4.000000,4.00000,4.000000,4.000000,4.000000,4.000000,...,2.000000,6.300000,5.900000,9.000000,1.360000,1.900000,2.010000,9.000000,0.987511,81.000000


In [17]:
judaism = df.loc[df['target_religion_jewish'] == True, ['comment_id', 'text', 'hate_speech_score']].drop_duplicates(subset='comment_id')
len(judaism)

1874

In [18]:
judaism.to_csv('data/ucberkeley-dlab_target_jewish.csv', index=False)
print(judaism['hate_speech_score'].describe())

count    1874.000000
mean       -0.857556
std         2.006435
min        -7.940000
25%        -2.180000
50%        -0.680000
75%         0.517500
max         5.090000
Name: hate_speech_score, dtype: float64


### Pilot Codebook Labeling

In [13]:
from dotenv import load_dotenv
import os
import anthropic
import json
import csv
import re
import pandas as pd
from config import UNIVERSAL, INPUT, I_1, I_2, I_3, I_4, I_5, I_NO_1, N_1, N_2, N_NO_1, J_1, J_2, J_3, J_NO_1

load_dotenv()
api_key = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=api_key)

In [ ]:
# Sanity Check (Sonnet across 3 runs for intra-model reliability)

# test ids and texts
test_ids = [29933, 39476, 40464, 985, 32861, 32448, 27527, 20045]
test_texts = {row['comment_id']: row['text'] for _, row in judaism[judaism['comment_id'].isin(test_ids)].iterrows()}

# prompt blocks (block_name, block_content, is_not)
blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

MODEL_NAME = "claude-sonnet-4-6"
N_RUNS = 3

# results[run][comment_id][block_name] = [(code_id, label), ...]
results = {r: {} for r in range(1, N_RUNS + 1)}

for run in range(1, N_RUNS + 1):
    for comment_id in test_ids:
        results[run][comment_id] = {}
        text = test_texts[comment_id]

        for block_name, block_content, is_not in blocks:
            is_or_is_not = "is NOT" if is_not else "IS"
            system_text = UNIVERSAL.format(ISorisNOT=is_or_is_not) + block_content
            user_text = INPUT.format(id=comment_id, text=text)

            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=1024,
                temperature=0,
                system=[
                    {
                        "type": "text",
                        "text": system_text,
                        "cache_control": {"type": "ephemeral"},
                    }
                ],
                messages=[{"role": "user", "content": user_text}],
            )

            raw = response.content[0].text.strip()

            try:
                clean = re.sub(r'```json|```', '', raw).strip()
                data = json.loads(clean)
                parsed = [tuple(pair) for pair in list(data.values())[0]]
                results[run][comment_id][block_name] = parsed
            except Exception as e:
                results[run][comment_id][block_name] = {"parse_error": str(e), "raw": raw}

# save raw json
with open("test/comparative_results.json", "w") as f:
    json.dump(results, f, indent=2)

# flatten to dataframe (comment_id, block, code_id, run, label)
rows = []
for run in range(1, N_RUNS + 1):
    for comment_id, blocks_dict in results[run].items():
        for block_name, labels in blocks_dict.items():
            if isinstance(labels, list):
                for item in labels:
                    if len(item) == 2:
                        code_id, label = item
                        rows.append([comment_id, block_name, code_id, run, label])
                    else:
                        rows.append([comment_id, block_name, "MALFORMED", run, str(item)])
            else:
                rows.append([comment_id, block_name, "PARSE_ERROR", run, str(labels)])

flat_df = pd.DataFrame(rows, columns=["comment_id", "block", "code_id", "run", "label"])
flat_df.to_csv("test/comparative_results_flat.csv", index=False)

print(f"Done. {len(flat_df)} rows saved to comparative_results_flat.csv and comparative_results.json")

In [24]:
# Direct comparison across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

pivot_df = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
).reset_index()

pivot_df.to_csv("test/comparative_pivot.csv", index=False)
print(pivot_df.to_string(index=False))

 comment_id  block                 code_id 1 2 3
        985    I_1                  D1HATE N N N
        985    I_1              D1MANIFEST N N N
        985    I_1            D1PERCEPTION A C A
        985    I_1       D2COLLECTIVEBLAME C C C
        985    I_1            D2CONSPIRACY N N N
        985    I_1          D2ISRAELTARGET N N N
        985    I_1            D2STEREOTYPE C C C
        985    I_2               E1RADICAL N N N
        985    I_2              E1VIOLENCE N N N
        985    I_2            E2ALLEGATION C C C
        985    I_2       E2COLLECTIVEPOWER N N N
        985    I_2            E2CONSPIRACY N N N
        985    I_2        E2CONTROLECONOMY N N N
        985    I_2            E2CONTROLGOV N N N
        985    I_2          E2CONTROLMEDIA N N N
        985    I_2          E2CONTROLOTHER N N N
        985    I_2        E2DEHUMANIZATION N N N
        985    I_2              E2DEMONIZE N N N
        985    I_2            E2STEREOTYPE C C C
        985    I_2  

In [25]:
# Intra-model agreement across runs

flat_df = pd.read_csv("test/comparative_results_flat.csv")

def pct_agreement(series_list):
    """Given a list of label-series aligned by index, return % where all match."""
    combined = pd.concat(series_list, axis=1)
    combined.columns = range(len(series_list))
    all_match = combined.apply(lambda row: row.nunique() == 1, axis=1)
    return all_match.mean() * 100

wide = flat_df.pivot_table(
    index=["comment_id", "block", "code_id"],
    columns="run",
    values="label",
    aggfunc="first"
)

run_cols = [c for c in wide.columns]
agreement_pct = pct_agreement([wide[c] for c in run_cols])

print(f"Sonnet: {agreement_pct:.2f}% full agreement across {len(run_cols)} runs")

pd.DataFrame([{"model": "sonnet", "in_model_agreement_pct": round(agreement_pct, 2)}]).to_csv(
    "test/in_model_agreement.csv", index=False
)

Sonnet: 98.62% full agreement across 3 runs


## Full Label

In [31]:
# CONSTRUCT BATCHES

import json
import os
import re

MODEL = "gpt-4o"
OUTPUT_DIR = "batch_pilot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

all_ids = judaism['comment_id'].tolist()
all_texts = {row['comment_id']: row['text'] for _, row in judaism.iterrows()}

# split comment ids into two halves
midpoint = len(all_ids) // 2
id_halves = [all_ids[:midpoint], all_ids[midpoint:]]

batch_files = []  # (block_name, half_idx, path)

for block_name, block_content, is_not in blocks:
    is_or_is_not = "is NOT" if is_not else "IS"
    for half_idx, id_subset in enumerate(id_halves):
        requests = []
        for comment_id in id_subset:
            text = all_texts[comment_id]
            prompt = (
                UNIVERSAL.format(ISorisNOT=is_or_is_not)
                + block_content
                + INPUT.format(id=comment_id, text=text)
            )
            custom_id = f"{block_name}__{comment_id}"
            requests.append({
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "response_format": {"type": "json_object"},
                    "max_tokens": 1024,
                    "temperature": 0,
                }
            })

        path = f"{OUTPUT_DIR}/batch_{block_name}_half{half_idx}.jsonl"
        with open(path, "w") as f:
            for r in requests:
                f.write(json.dumps(r) + "\n")
        batch_files.append((block_name, half_idx, path))
        print(f"Wrote {len(requests)} requests to {path}")

print(f"\nTotal batch files: {len(batch_files)}")

Wrote 937 requests to batch_pilot/batch_I_1_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_1_half1.jsonl
Wrote 937 requests to batch_pilot/batch_I_2_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_2_half1.jsonl
Wrote 937 requests to batch_pilot/batch_I_3_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_3_half1.jsonl
Wrote 937 requests to batch_pilot/batch_I_4_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_4_half1.jsonl
Wrote 937 requests to batch_pilot/batch_I_5_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_5_half1.jsonl
Wrote 937 requests to batch_pilot/batch_I_NO_1_half0.jsonl
Wrote 937 requests to batch_pilot/batch_I_NO_1_half1.jsonl
Wrote 937 requests to batch_pilot/batch_N_1_half0.jsonl
Wrote 937 requests to batch_pilot/batch_N_1_half1.jsonl
Wrote 937 requests to batch_pilot/batch_N_2_half0.jsonl
Wrote 937 requests to batch_pilot/batch_N_2_half1.jsonl
Wrote 937 requests to batch_pilot/batch_N_NO_1_half0.jsonl
Wrote 937 requests to batch_pilot/batch

In [ ]:
# SUBMIT AND COMPLETE BATCHES SEQUENTIALLY

import time

batch_job_map = {}

for block_name, half_idx, path in batch_files:
    with open(path, "rb") as f:
        uploaded = client.files.create(file=f, purpose="batch")
    batch = client.batches.create(
        input_file_id=uploaded.id,
        endpoint="/v1/chat/completions",
        completion_window="24h",
    )
    key = f"{block_name}_half{half_idx}"
    batch_job_map[key] = batch.id
    print(f"Submitted {key}: {batch.id} (status: {batch.status})")

    # wait until this batch clears the queue before submitting the next
    while True:
        current = client.batches.retrieve(batch.id)
        if current.status in ("completed", "failed", "expired", "cancelled"):
            print(f"  {key} finished with status: {current.status}")
            break
        time.sleep(30)

with open(f"{OUTPUT_DIR}/batch_job_map.json", "w") as f:
    json.dump(batch_job_map, f, indent=2)

print("\nAll batches submitted and completed sequentially.")

Submitted I_1_half0: batch_6a317c6171cc8190958301d6f24d15c5 (status: validating)
  I_1_half0 finished with status: failed
Submitted I_1_half1: batch_6a317c8219e08190a14ab79ce2b7acd3 (status: validating)


In [11]:
import json
import os
import tiktoken

blocks = [
    ("I_1", I_1, False),
    ("I_2", I_2, False),
    ("I_3", I_3, False),
    ("I_4", I_4, False),
    ("I_5", I_5, False),
    ("I_NO_1", I_NO_1, True),
    ("N_1", N_1, False),
    ("N_2", N_2, False),
    ("N_NO_1", N_NO_1, True),
    ("J_1", J_1, False),
    ("J_2", J_2, False),
    ("J_3", J_3, False),
    ("J_NO_1", J_NO_1, True),
]

enc = tiktoken.encoding_for_model("gpt-4o")

OUTPUT_DIR = "batch_pilot"

universal_tokens_sample = len(enc.encode(UNIVERSAL.format(ISorisNOT="IS")))

print(f"UNIVERSAL alone: {universal_tokens_sample:,} tokens")
print()

grand_total_tokens = 0
grand_universal_tokens = 0  # universal portion summed across every request, every file
grand_block_tokens = 0      # universal + block portion summed across every request
grand_other_tokens = 0      # the input/comment-specific portion

per_block_totals = {}  # block_name -> {"universal": x, "universal_plus_block": y, "other": z, "count": n}

for fname in sorted(os.listdir(OUTPUT_DIR)):
    if not fname.endswith(".jsonl") or fname == "batch_job_map.json":
        continue

    # extract block name from filename: batch_{block_name}_chunk{n}.jsonl
    block_name = fname.replace("batch_", "").split("_half")[0]

    if block_name not in per_block_totals:
        # find matching block content
        match = next((bc for bn, bc, is_not in blocks if bn == block_name), None)
        is_not = next((isn for bn, bc, isn in blocks if bn == block_name), False)
        is_or_is_not = "is NOT" if is_not else "IS"
        universal_str = UNIVERSAL.format(ISorisNOT=is_or_is_not)
        universal_tok = len(enc.encode(universal_str))
        universal_plus_block_tok = len(enc.encode(universal_str + match)) if match else universal_tok
        per_block_totals[block_name] = {
            "universal_per_request": universal_tok,
            "universal_plus_block_per_request": universal_plus_block_tok,
            "block_only_per_request": universal_plus_block_tok - universal_tok,
            "n_requests": 0,
            "total_prompt_tokens": 0,
        }

    path = os.path.join(OUTPUT_DIR, fname)
    with open(path) as f:
        for line in f:
            record = json.loads(line)
            content = record["body"]["messages"][0]["content"]
            n_tok = len(enc.encode(content))
            per_block_totals[block_name]["n_requests"] += 1
            per_block_totals[block_name]["total_prompt_tokens"] += n_tok

# compute aggregates
print(f"{'Block':<10} {'Requests':>9} {'Universal/req':>14} {'Block-only/req':>15} {'Other/req (avg)':>17} {'Total tokens':>13}")
grand_universal_total = 0
grand_blockonly_total = 0
grand_other_total = 0
grand_total = 0

for block_name, d in per_block_totals.items():
    n = d["n_requests"]
    universal_total = d["universal_per_request"] * n
    blockonly_total = d["block_only_per_request"] * n
    avg_other_per_req = (d["total_prompt_tokens"] - d["universal_plus_block_per_request"] * n) / n if n else 0
    other_total = d["total_prompt_tokens"] - universal_total - blockonly_total

    grand_universal_total += universal_total
    grand_blockonly_total += blockonly_total
    grand_other_total += other_total
    grand_total += d["total_prompt_tokens"]

    print(f"{block_name:<10} {n:>9} {d['universal_per_request']:>14} {d['block_only_per_request']:>15} {avg_other_per_req:>17.1f} {d['total_prompt_tokens']:>13,}")

print()
print(f"GRAND TOTALS across all files:")
print(f"  Universal instructions: {grand_universal_total:,} tokens ({grand_universal_total/grand_total*100:.1f}%)")
print(f"  Block-specific codes:   {grand_blockonly_total:,} tokens ({grand_blockonly_total/grand_total*100:.1f}%)")
print(f"  Comment/ID (other):     {grand_other_total:,} tokens ({grand_other_total/grand_total*100:.1f}%)")
print(f"  TOTAL:                  {grand_total:,} tokens")

UNIVERSAL alone: 712 tokens

Block       Requests  Universal/req  Block-only/req   Other/req (avg)  Total tokens
I_1             1874            712             185              63.1     1,799,285
I_2             1874            712             280              63.1     1,977,315
I_3             1874            712             194              63.1     1,816,151
I_4             1874            712             312              63.1     2,037,283
I_5             1874            712             217              63.1     1,859,253
I_NO_1          1874            713              28              63.1     1,506,941
J_1             1874            712             386              63.1     2,175,959
J_2             1874            712             189              63.1     1,806,781
J_3             1874            712             235              63.1     1,892,985
J_NO_1          1874            713             516              63.1     2,421,453
N_1             1874            712            

In [ ]:
# STORE

all_results = []

with open(f"{OUTPUT_DIR}/batch_job_map.json") as f:
    batch_job_map = json.load(f)

for key, job_id in batch_job_map.items():
    batch = client.batches.retrieve(job_id)
    if batch.status != "completed":
        print(f"{key} not yet complete: {batch.status}")
        continue

    output_file = client.files.content(batch.output_file_id)
    lines = output_file.text.strip().split("\n")

    for line in lines:
        record = json.loads(line)
        custom_id = record["custom_id"]
        block_name_parsed, comment_id = custom_id.split("__")

        if record["response"]["status_code"] == 200:
            raw = record["response"]["body"]["choices"][0]["message"]["content"]
            try:
                data = json.loads(raw)
                parsed = [tuple(pair) for pair in list(data.values())[0]]
                for code_id, label in parsed:
                    all_results.append({
                        "block": block_name_parsed,
                        "comment_id": comment_id,
                        "code_id": code_id,
                        "label": label,
                    })
            except Exception as e:
                all_results.append({
                    "block": block_name_parsed,
                    "comment_id": comment_id,
                    "code_id": "PARSE_ERROR",
                    "label": str(e),
                })
        else:
            all_results.append({
                "block": block_name_parsed,
                "comment_id": comment_id,
                "code_id": "API_ERROR",
                "label": str(record["response"]["status_code"]),
            })

results_df = pd.DataFrame(all_results)
results_df.to_csv(f"{OUTPUT_DIR}/full_batch_results.csv", index=False)
results_df.to_json(f"{OUTPUT_DIR}/full_batch_results.json", orient="records", indent=2)

summary = results_df.groupby('block').size().reset_index(name='rows_returned')
print(summary.to_string(index=False))
print(f"\nSaved {len(results_df)} total rows to {OUTPUT_DIR}/full_batch_results.csv and .json")

## Create Features

In [ ]:
# One-hot encode labels

In [ ]:
# Create folds

In [ ]:
# embed text

# PCA?

### Predict UC Berkeley Discrete Values Using Label and Semantic Features